# VM.AI Regressors - Difficulty & Importance Prediction
## Why Separate Regressors?
### The Problem: T5 Struggles with Numeric Fields
Throughout testing, we kept running into the same issue with T5 predicting `difficulty` and `importance`:
- **Semantic mismatch** - words like `"extreme"` would output `0.5`, while `"not difficult"` would output `0.85`
- **Binary-like outputs** - the model consistently mapped `"easy"` to `~0.15` and `"hard"` to `~0.85` with nothing in between
- **No per-task context** - `"hard homework"` and `"hard gym session"` got the same difficulty score even though they're totally different
- **Importance/difficulty confusion** - couldn't tell that `"hard homework"` might be less physically difficult but more important than `"hard gym session"`
We tried fixing this across multiple parser versions - changed architectures, restructured data, iterated on prompts. Same problem every time. The issue is fundamental: **T5 is a text-to-text model, not a regressor**. It generates tokens, not numbers. When trained to predict floats like `0.72`, it just learns surface-level token patterns instead of actual numeric relationships.
More training data would help a bit, but it doesn't fix the architectural mismatch.
### The Solution
We split the job: **dedicated XGBoost regressors** on top of **SentenceTransformer embeddings**:
- **SentenceTransformer** (`all-MiniLM-L6-v2`) converts text into 384-dim semantic embeddings that actually understand context and nuance
- **Two independent XGBoostRegressor models** predict `difficulty` and `importance` as real continuous floats in `[0, 1]`
- **Modular design** - regressors can be updated independently of the T5 parser
## Training Requirements
This notebook is nearly self-contained. It needs:
- **`src/parser/yaml_parser.py`** - parses YAML datasets into usable format
- **`src/parser/vars.py`** - shared constants (imported by `yaml_parser.py`)
- **`data/` folder** - training datasets (`VMAI_REAL_Data.yaml`, `VMAI_REGR_Data.yaml`, etc.)
- **Optional custom datasets** - extra labeled examples for edge cases or underrepresented task types
## Data Source
Training data comes from `data/VMAI_REAL_Data.yaml` + `data/VMAI_REGR_Data.yaml` - hand-labeled real user prompts with explicit `difficulty` and `importance` values. Merged into a single CSV for training.
## Train/Test Split
80/20 split with `random_state=42`. Models are evaluated on unseen test data for honest MAE and RВІ.
## Integration
Trained models are saved as `.pkl` files and bundled with the T5 parser on HuggingFace (`vaneaa/vmai-parser`). At runtime, `TaskPlannerPredictor` loads both regressors and applies them during `predict_add()` and `predict_modify()`.
## Interpreting Predictions
**Difficulty (0→1):** trivial → extremely hard  
**Importance (0→1):** optional → critical
### Subjectivity
Difficulty and importance are inherently personal. A task you find "hard" might feel "moderate" to someone else. The model learned your specific labeling patterns. Predictions may not generalize to other users with different thresholds for what counts as "hard" or "important."
### Performance
- **R² ≈ 0.68 (diff) / 0.58 (imp)** — the model captures ~⅔ of your difficulty ratings and ~½ of importance ratings. The remaining ~30-40% reflects noise from personal inconsistency over 500+ labels plus factors the text alone can't convey (energy, deadlines, mood).
- **MAE ≈ 0.10–0.12** — predictions are off by about 10 points on a 0–100 scale.
### When to Trust It
- Clear language with explicit difficulty/importance keywords: *"urgent server fix"*, *"easy stretch"*
- Routine tasks with consistent phrasing across your dataset
### When to Override
- Tasks where context matters more than text: *"finish report"* could be easy or crushing depending on your deadline
- The task feels obviously misjudged — your intuition beats the model

In [1]:
import sys, re, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from yaml_parser import VMAI_RealDataParser
from sentence_transformers import SentenceTransformer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression, RidgeCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics.pairwise import cosine_similarity
import joblib
from datetime import datetime
import os
from collections import defaultdict
warnings.filterwarnings("ignore")


In [2]:
# Consistent across project
plt.style.use("dark_background")
plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor": "#0d1117",
    "axes.edgecolor": "#30363d",
    "axes.labelcolor": "#c9d1d9",
    "text.color": "#c9d1d9",
    "xtick.color": "#8b949e",
    "ytick.color": "#8b949e",
    "grid.color": "#21262d",
})

In [3]:
from datetime import datetime
import os

def save_fig(name):
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    os.makedirs("plots", exist_ok=True)
    plt.savefig(f"plots/{name}_{timestamp}.png", dpi=150, bbox_inches="tight")
    print(f"  Saved: plots/{name}_{timestamp}.png")


In [ ]:
train = pd.read_csv("VMAI_REGR_Data.csv")

print("Shape:", train.shape)
print("\nMissing values:\n", train.isnull().sum())
print("\nDifficulty stats:\n", train["difficulty"].describe())
print("\nImportance stats:\n", train["importance"].describe())

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Difficulty distribution
axes[0, 0].hist(train["difficulty"], bins=20, edgecolor="#30363d", alpha=0.8, color="#58a6ff")
axes[0, 0].set_title("Difficulty Distribution", fontweight="bold")
axes[0, 0].set_xlabel("Difficulty")
axes[0, 0].set_ylabel("Count")
axes[0, 0].grid(True, alpha=0.15)

# Importance distribution
axes[0, 1].hist(train["importance"], bins=20, edgecolor="#30363d", alpha=0.8, color="#bc8cff")
axes[0, 1].set_title("Importance Distribution", fontweight="bold")
axes[0, 1].set_xlabel("Importance")
axes[0, 1].set_ylabel("Count")
axes[0, 1].grid(True, alpha=0.15)

# Box plot comparison
box_data = [train["difficulty"], train["importance"]]
axes[1, 0].boxplot(box_data, labels=["Difficulty", "Importance"], patch_artist=True,
                    boxprops=dict(facecolor="#21262d", edgecolor="#58a6ff", linewidth=1.5),
                    medianprops=dict(color="#f78166", linewidth=2),
                    whiskerprops=dict(color="#8b949e"),
                    capprops=dict(color="#8b949e"),
                    flierprops=dict(marker="o", markerfacecolor="#f78166", markersize=4))
axes[1, 0].set_title("Distribution Comparison", fontweight="bold")
axes[1, 0].set_ylabel("Score")
axes[1, 0].grid(True, alpha=0.15, axis="y")

# KDE overlay
train["difficulty"].plot.kde(ax=axes[1, 1], label="Difficulty", color="#58a6ff", linewidth=2)
train["importance"].plot.kde(ax=axes[1, 1], label="Importance", color="#bc8cff", linewidth=2)
axes[1, 1].set_title("Density Overlay", fontweight="bold")
axes[1, 1].set_xlabel("Score")
axes[1, 1].set_xlim(-0.1, 1.1)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.15)

plt.tight_layout()
save_fig("data")
plt.show()


In [ ]:
X = train["text"].values
y_diff = train["difficulty"].values
y_imp = train["importance"].values

bins = pd.qcut(y_diff, q=5, labels=False, duplicates='drop')
X_train, X_test, y_diff_train, y_diff_test, y_imp_train, y_imp_test, _, test_idx = train_test_split(
    X, y_diff, y_imp, train.index, test_size=0.2, random_state=42, stratify=bins
)

print(f"Train: {len(X_train)} samples")
print(f"Test:  {len(X_test)} samples")
print(f"Diff train range: {y_diff_train.min():.2f}-{y_diff_train.max():.2f}")
print(f"Diff test range:  {y_diff_test.min():.2f}-{y_diff_test.max():.2f}")

In [ ]:
encoder = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
encoder = SentenceTransformer('all-mpnet-base-v2')

In [ ]:
emb_train = encoder.encode(X_train, show_progress_bar=False)
emb_test = encoder.encode(X_test, show_progress_bar=False)
print(f"Train: {emb_train.shape}  Test: {emb_test.shape}")

In [ ]:
URGENT = {"urgent", "asap", "critical", "deadline", "important", "immediately"}
HARD = {"hard", "difficult", "complex", "tough", "challenging", "heavy", "intense"}
EASY = {"easy", "simple", "quick", "light", "trivial", "basic", "gentle"}
TIME = {"minute", "hour", "day", "week", "month", "today", "tomorrow"}
NEGATION = {"not", "no", "never", "without", "n't"}
INTENSIFIERS = {"very", "extremely", "super", "incredibly", "insanely", "really", "highly", "especially", "particularly"}
HEDGES = {"kind of", "sort of", "somewhat", "fairly", "rather", "pretty", "quite", "slightly", "barely"}

def _is_contextually_negated(words, i):
    for j in range(max(0, i - 2), i):
        if words[j] in NEGATION or words[j].endswith("n't"):
            return True
    return False

def extract_lexical_features(texts):
    feats = []
    for t in texts:
        w = t.lower().split()
        n_chars = len(t)
        n_words = len(w)

        urgent_raw = hard_raw = easy_raw = time_raw = 0
        urgent_neg = hard_neg = easy_neg = time_neg = 0
        negation_count = intense_count = 0

        for i, x in enumerate(w):
            if x in NEGATION or x.endswith("n't"):
                negation_count += 1
            if x in INTENSIFIERS:
                intense_count += 1
            negated = _is_contextually_negated(w, i)
            if x in URGENT:
                if negated:
                    urgent_neg += 1
                else:
                    urgent_raw += 1
            if x in HARD:
                if negated:
                    hard_neg += 1
                else:
                    hard_raw += 1
            if x in EASY:
                if negated:
                    easy_neg += 1
                else:
                    easy_raw += 1
            if x in TIME:
                if negated:
                    time_neg += 1
                else:
                    time_raw += 1

        full_lower = t.lower()
        hedge_count = sum(1 for h in HEDGES if h in full_lower)

        feats.append([
            n_words, n_chars, n_chars / max(n_words, 1),
            t.count("!"),
            sum(1 for c in t if c.isupper()),
            sum(1 for c in t if c.isupper()) / max(n_chars, 1),
            urgent_raw, hard_raw, easy_raw, time_raw,
            urgent_neg, hard_neg, easy_neg, time_neg,
            negation_count, intense_count, hedge_count,
            int(bool(re.search(r"\d+", t))),
        ])
    return np.array(feats)

feat_train = extract_lexical_features(X_train)
feat_test = extract_lexical_features(X_test)

# TF-IDF features
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=800, lowercase=True)
tfidf_train = tfidf.fit_transform(X_train)
tfidf_test = tfidf.transform(X_test)

svd = TruncatedSVD(n_components=100, random_state=42)
tfidf_svd_train = svd.fit_transform(tfidf_train)
tfidf_svd_test = svd.transform(tfidf_test)

X_train_full = np.concatenate([emb_train, feat_train, tfidf_svd_train], axis=1)
X_test_full = np.concatenate([emb_test, feat_test, tfidf_svd_test], axis=1)

print(f"Embeddings: {emb_train.shape[1]} + Lexical: {feat_train.shape[1]} + TF-IDF(SVD): {tfidf_svd_train.shape[1]} = {X_train_full.shape[1]} total")


In [9]:
def train_models(model_set, name, X, y):
    print(f"Training {name} models")
    for model_name in model_set:
        print(f"  Training '{model_name}'...", end=' ', flush=True)
        model_set[model_name].fit(X, y)
        print("done")

def eval_model(model, X, y, model_name=""):
    pred = np.clip(model.predict(X), 0, 1)
    mae = mean_absolute_error(y, pred)
    r2 = r2_score(y, pred)
    baseline = mean_absolute_error(y, np.full_like(y, np.mean(y)))
    return {
        "model": model_name,
        "MAE": round(mae, 4),
        "R2": round(r2, 4),
        "baseline_MAE": round(baseline, 4),
        "improvement_pct": round((baseline - mae) / baseline * 100, 1),
    }

def eval_models(model_set, name, X, y):
    print(f"Evaluating {name} models")
    results = []
    for model_name in model_set:
        r = eval_model(model_set[model_name], X, y, model_name)
        results.append(r)
        print(f"  {model_name:20s}  MAE={r['MAE']}  R\u00b2={r['R2']}")
    return results

def plot_model_comparison(results, title):
    names = [r["model"] for r in results]
    maes = [r["MAE"] for r in results]
    r2s = [r["R2"] for r in results]
    x = range(len(names))
    plt.figure(figsize=(10, 4))
    plt.bar([i - 0.2 for i in x], maes, 0.4, label='MAE', color='coral')
    plt.bar([i + 0.2 for i in x], r2s, 0.4, label='R\u00b2', color='steelblue')
    plt.xticks(x, names, rotation=45, ha='right')
    plt.title(title)
    plt.axhline(0, color='gray', linestyle='--', linewidth=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_regression_diagnostics(model, X, y, title=""):
    pred = np.clip(model.predict(X), 0, 1)
    residuals = y - pred

    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    fig.suptitle(title, fontsize=14)

    axes[0, 0].scatter(y, pred, alpha=0.5, edgecolors='k', linewidth=0.5)
    axes[0, 0].plot([0, 1], [0, 1], 'r--', linewidth=1)
    axes[0, 0].set_xlabel('Actual')
    axes[0, 0].set_ylabel('Predicted')
    axes[0, 0].set_title('Actual vs Predicted')
    axes[0, 0].set_xlim(0, 1); axes[0, 0].set_ylim(0, 1)
    axes[0, 0].axis('equal')

    axes[0, 1].hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='coral')
    axes[0, 1].axvline(0, color='red', linestyle='--', linewidth=1)
    axes[0, 1].set_xlabel('Residual (Actual - Predicted)')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].set_title(f'Residuals  (MAE={mean_absolute_error(y, pred):.4f})')

    axes[1, 0].scatter(pred, residuals, alpha=0.5, edgecolors='k', linewidth=0.5)
    axes[1, 0].axhline(0, color='red', linestyle='--', linewidth=1)
    axes[1, 0].set_xlabel('Predicted')
    axes[1, 0].set_ylabel('Residual')
    axes[1, 0].set_title('Residuals vs Predicted')

    bins = np.linspace(0, 1, 11)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    bin_indices = np.digitize(pred, bins) - 1
    mae_per_bin = [np.mean(np.abs(residuals[bin_indices == i])) if np.sum(bin_indices == i) > 0 else 0 for i in range(len(bins) - 1)]
    axes[1, 1].bar(bin_centers, mae_per_bin, width=0.08, edgecolor='black', alpha=0.7, color='steelblue')
    axes[1, 1].set_xlabel('Prediction Bin')
    axes[1, 1].set_ylabel('Mean Absolute Error')
    axes[1, 1].set_title(f'Error by Prediction Range')

    save_fig(title.lower())

    plt.tight_layout()
    plt.show()

    mae = mean_absolute_error(y, pred)
    r2 = r2_score(y, pred)
    print(f"  MAE={mae:.4f}  R\u00b2={r2:.4f}")


def fit_calibration(model, X_train, y_train, X_val, y_val):
    train_pred = np.clip(model.predict(X_train), 0, 1)
    val_pred = np.clip(model.predict(X_val), 0, 1)
    cal = IsotonicRegression(out_of_bounds='clip')
    cal.fit(val_pred, y_val)
    residual_std = np.std(y_train - train_pred)
    return cal, residual_std


def evaluate_cv(model, X, y, cv=5):
    preds = cross_val_predict(model, X, y, cv=cv)
    mae = mean_absolute_error(y, preds)
    r2 = r2_score(y, preds)
    print(f"  {cv}-fold CV  MAE={mae:.4f}  R\u00b2={r2:.4f}")
    return mae, r2


def detect_data_leakage(X_train_texts, X_test_texts, threshold=0.9):
    vec = TfidfVectorizer(lowercase=True, max_features=5000)
    tfidf_all = vec.fit_transform(list(X_train_texts) + list(X_test_texts))
    n_train = len(X_train_texts)
    sims = cosine_similarity(tfidf_all[:n_train], tfidf_all[n_train:])
    flagged = np.sum(sims > threshold)
    print(f"  Near-duplicate pairs (cosine > {threshold}): {flagged}")
    return flagged


def predict_with_intervals(model, X, residual_std):
    pred = np.clip(model.predict(X), 0, 1)
    lo = np.clip(pred - 1.96 * residual_std, 0, 1)
    hi = np.clip(pred + 1.96 * residual_std, 0, 1)
    return pred, lo, hi


def log_uncertain_predictions(texts, preds, uncertainties, output_csv="uncertain_predictions.csv", threshold=0.15):
    import csv
    flagged = [(t, p, u) for t, p, u in zip(texts, preds, uncertainties) if u > threshold]
    print(f"  High-uncertainty predictions (std > {threshold}): {len(flagged)}/{len(texts)}")
    if flagged:
        exists = os.path.exists(output_csv)
        with open(output_csv, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            if not exists:
                writer.writerow(["timestamp", "text", "prediction", "uncertainty"])
            for t, p, u in flagged:
                writer.writerow([datetime.now().isoformat(), t, f"{p:.3f}", f"{u:.4f}"])
    print(f"  Appended to {output_csv}")


#### Old versions of this notebook included a hyperparameter tuner for XGBRegressor and these were the results The amount of params in the `param_grid` were enough to get the best parameters (5 different paramter values per paramter and 5 CV + Gridsearch so each combination was tested)


#### With testing tho, it became clear that there were better alternatives (like RigdeCV) and hyperparameter tuning was no longer required

In [10]:
best_params = {'subsample': 0.6, 'n_estimators': 500, 'min_child_weight': 7, 'max_depth': 6, 'learning_rate': 0.01, 'gamma': 0, 'colsample_bytree': 0.6}

In [11]:
MODELS = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(max_iter=5000),
    "ElasticNet": ElasticNet(max_iter=5000),
    "KNN (k=7)": KNeighborsRegressor(n_neighbors=7),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Extra Trees": ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "SVR (RBF)": SVR(kernel='rbf', gamma='scale'),
    "SVR (linear)": SVR(kernel='linear'),
    "GBoost": GradientBoostingRegressor(n_estimators=200, random_state=42),
    "MLP (64,32)": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=200, random_state=42, verbose=-1),
    "XGB": XGBRegressor(),
    "XGB (tuned)": XGBRegressor(**best_params, random_state=42),
}

diff_models = {k: v for k, v in MODELS.items()}
imp_models = {k: v for k, v in MODELS.items()}

## Cross-Validation & Data Leakage Check
5-fold CV for honest performance estimate, plus cosine-similarity leakage detection.

In [ ]:
# 5-fold Cross-Validation
print("5-Fold Cross-Validation:")
diff_cv_mae, diff_cv_r2 = evaluate_cv(Ridge(), X_train_full, y_diff_train, cv=5)
imp_cv_mae, imp_cv_r2 = evaluate_cv(Ridge(), X_train_full, y_imp_train, cv=5)

# Data leakage check
print("\nData Leakage Check:")
detect_data_leakage(X_train, X_test, threshold=0.9)


In [13]:
def plot_final_diagnostics():
  plot_regression_diagnostics(diff_final, X_test_full, y_diff_test, title="Difficulty")
  plot_regression_diagnostics(imp_final, X_test_full, y_imp_test, title="Importance")


In [ ]:
train_models(diff_models, "difficulty", X_train_full, y_diff_train)
diff_results = eval_models(diff_models, "difficulty", X_test_full, y_diff_test)
plot_model_comparison(diff_results, "Difficulty вЂ” Model Comparison")

In [ ]:
train_models(imp_models, "importance", X_train_full, y_imp_train)
imp_results = eval_models(imp_models, "importance", X_test_full, y_imp_test)
plot_model_comparison(imp_results, "Importance вЂ” Model Comparison")

# The following cells represent the different models used for for regression and in what order

- XGBRegressor
- Ridge
- RidgeCV

In [ ]:
diff_final = XGBRegressor(**best_params, random_state=42, device="cuda")
diff_final.fit(X_train_full, y_diff_train)

imp_final = XGBRegressor(**best_params, random_state=42, device="cuda" )
imp_final.fit(X_train_full, y_imp_train)

In [ ]:
plot_final_diagnostics()

In [ ]:
diff_final = Ridge()
diff_final.fit(X_train_full, y_diff_train)

imp_final = Ridge()
imp_final.fit(X_train_full, y_imp_train)

In [ ]:
plot_final_diagnostics()

In [ ]:
diff_final = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100], store_cv_values=True)
diff_final.fit(X_train_full, y_diff_train)
print(f"Best alpha (diff): {diff_final.alpha_}")

imp_final = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100], store_cv_values=True)
imp_final.fit(X_train_full, y_imp_train)
print(f"Best alpha (imp): {imp_final.alpha_}")

In [ ]:
plot_final_diagnostics()

In [ ]:
joblib.dump(diff_final, 'difficulty_regressor.pkl')
joblib.dump(imp_final, 'importance_regressor.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(svd, 'tfidf_svd.pkl')

In [ ]:
def predict_task(texts, encoder, diff_model, imp_model, cal_diff=None, cal_imp=None, residual_std=None):
    if isinstance(texts, str):
        texts = [texts]
    emb = encoder.encode(texts, show_progress_bar=False)
    feats = extract_lexical_features(texts)
    tfidf_feat = tfidf.transform(texts)
    tfidf_svd_feat = svd.transform(tfidf_feat)
    X = np.concatenate([emb, feats, tfidf_svd_feat], axis=1)
    d = np.clip(diff_model.predict(X), 0, 1)
    i = np.clip(imp_model.predict(X), 0, 1)
    if cal_diff is not None:
        d = cal_diff.predict(d)
    if cal_imp is not None:
        i = cal_imp.predict(i)
    if residual_std is not None:
        for t, dv, iv in zip(texts, d, i):
            dlo = max(0, dv - 1.96 * residual_std)
            dhi = min(1, dv + 1.96 * residual_std)
            ilo = max(0, iv - 1.96 * residual_std)
            ihi = min(1, iv + 1.96 * residual_std)
            print(f"{t:<60} {dv:.3f} {iv:.3f} {dlo:.3f}-{dhi:.3f} {ilo:.3f}-{ihi:.3f}")
    else:
        for t, dv, iv in zip(texts, d, i):
            print(f"{t:<60} diff={dv:.3f}  imp={iv:.3f}")

custom = [
    "finish project report by friday",
    "buy groceries later",
    "urgent server outage fix",
    "read a book before bed",
    "hard workout",
    "easy stretch",
]
predict_task(custom, encoder, diff_final, imp_final)


In [ ]:
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import files
    files.download('difficulty_regressor.pkl')
    files.download('importance_regressor.pkl')
    files.download('tfidf_vectorizer.pkl')
    files.download('tfidf_svd.pkl')
print("Models saved to disk.")

In [ ]:
# Active learning: log uncertain predictions from test set
print("Uncertainty Analysis on Test Set:")
test_preds = np.clip(diff_final.predict(X_test_full), 0, 1)
test_uncertainties = np.abs(y_diff_test - test_preds)
log_uncertain_predictions(X_test[:20], test_preds[:20], test_uncertainties[:20], output_csv="uncertain_predictions.csv")


In [ ]:
import re
MODIFY_PATTERNS = re.compile(r"make it|change|set to|move|push|add |remove|rename|categorize|clear |cancel|mark it|extend|shorten|reschedule", re.I)

test_indices = X_test_index if 'X_test_index' in dir() else train.index[-len(X_test):]
is_mod_test = train.loc[test_indices, 'text'].str.contains(MODIFY_PATTERNS).values

add_mask = ~is_mod_test
mod_mask = is_mod_test

print(f"Add: {add_mask.sum()}  Modify: {mod_mask.sum()}\n")
for target, y_test, model in [("Difficulty", y_diff_test, diff_final), ("Importance", y_imp_test, imp_final)]:
    print(f"=== {target} ===")
    for label, mask in [("Add-only", add_mask), ("Modify-only", mod_mask)]:
        if mask.sum() == 0:
            continue
        pred = np.clip(model.predict(X_test_full[mask]), 0, 1)
        mae = mean_absolute_error(y_test[mask], pred)
        r2 = r2_score(y_test[mask], pred)
        print(f"  {label:15s} ({mask.sum():3d})  MAE={mae:.4f}  R²={r2:.4f}")
    print()

In [ ]:
while True:
    text = input(">> ")
    if text in ("exit", "quit"):
        break
    predict_task(text, encoder, diff_final, imp_final)